# Reading Data — Lab

**Course:** Machine Learning Foundation  
**Topic:** Praktik membaca database baseball.

Notebook ini menyelesaikan latihan membaca tabel `allstarfull` dari `baseball.db`, lalu melakukan analisis sederhana.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

DB_PATH = Path("data/baseball.db")
if not DB_PATH.exists():
    raise FileNotFoundError(f"Database tidak ditemukan: {DB_PATH.resolve()}")
connection = sqlite3.connect(DB_PATH)

## 1. Menampilkan tabel database

Periksa tabel yang tersedia sebelum menentukan tabel yang akan dianalisis.

In [2]:
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;", connection
)
tables

,name
0,allstarfull
1,batting
2,schools


## 2. Membaca seluruh data All-Star

Tabel `allstarfull` berisi catatan partisipasi pemain pada pertandingan All-Star. Kolom `playerID` merupakan ID pemain.

In [3]:
query = "SELECT * FROM allstarfull;"
allstar = pd.read_sql_query(query, connection)
print("Ukuran data:", allstar.shape)
display(allstar.head())

Ukuran data: (5148, 9)


,index,playerID,yearID,gameNum,gameID,teamID,lgID,GP,startingPos
0,0,gomezle01,1933,0,ALS193307060,NYA,AL,1.0,1.0
1,1,ferreri01,1933,0,ALS193307060,BOS,AL,1.0,2.0
2,2,gehrilo01,1933,0,ALS193307060,NYA,AL,1.0,3.0
3,3,gehrich01,1933,0,ALS193307060,DET,AL,1.0,4.0
4,4,dykesji01,1933,0,ALS193307060,CHA,AL,1.0,5.0


## 3. Menghitung jumlah penampilan per pemain

Kita mengelompokkan berdasarkan `playerID` dan menjumlahkan `GP`. Perhatikan bahwa `GP` adalah kolom dari dataset ini; hasilnya adalah peringkat berdasarkan total nilai GP yang tercatat, bukan penilaian menyeluruh atas kemampuan pemain.

In [4]:
player_summary_query = """
SELECT playerID,
       SUM(GP) AS total_gp,
       COUNT(*) AS allstar_records,
       AVG(startingPos) AS avg_starting_position
FROM allstarfull
GROUP BY playerID
ORDER BY total_gp DESC, allstar_records DESC
LIMIT 10;
"""
player_summary = pd.read_sql_query(player_summary_query, connection)
display(player_summary)

,playerID,total_gp,allstar_records,avg_starting_position
0,aaronha01,24.0,25,8.470588
1,mayswi01,24.0,24,8.000000
2,musiast01,24.0,24,6.357143
3,ripkeca01,18.0,19,5.823529
4,willite01,18.0,19,7.000000
5,robinbr01,18.0,18,5.000000
6,mantlmi01,16.0,20,8.076923
7,kalinal01,16.0,18,8.714286
8,rosepe01,16.0,17,5.375000
9,berrayo01,15.0,18,2.000000


## 4. Latihan tambahan: ringkasan per tahun

Contoh analisis lain adalah menghitung jumlah baris dan pemain unik yang tercatat pada setiap tahun.

In [5]:
year_summary = pd.read_sql_query("""
SELECT yearID,
       COUNT(*) AS records,
       COUNT(DISTINCT playerID) AS unique_players
FROM allstarfull
GROUP BY yearID
ORDER BY yearID;
""", connection)
display(year_summary.head(10))

,yearID,records,unique_players
0,1933,36,36
1,1934,41,41
2,1935,41,41
3,1936,43,43
4,1937,48,48
5,1938,48,48
6,1939,51,51
7,1940,53,53
8,1941,52,52
9,1942,53,53


## 5. Kesimpulan

- Database dapat diperiksa melalui `sqlite_master`.
- Query SQL mengambil kolom dan baris yang diperlukan.
- `GROUP BY`, `SUM`, `COUNT`, dan `AVG` digunakan untuk membuat ringkasan.
- Interpretasi hasil harus sesuai dengan definisi kolom dalam dataset.

In [6]:
connection.close()
print("Koneksi database ditutup.")

Koneksi database ditutup.
